# 🦠 COVID-19 Global Data Analysis
**Author:** Ahmed Walid  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn  
**Goal:** Analyze COVID-19 infection rates, mortality trends, and vaccination progress across countries and time periods.


## 1. 📦 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
sns.set_theme(style='darkgrid', palette='muted')
print('✅ Libraries loaded successfully')

## 2. 🗄️ Generate COVID-19 Dataset

In [ ]:
np.random.seed(42)

countries = {
    'USA':    {'pop': 331e6, 'vax_rate': 0.68},
    'India':  {'pop': 1380e6,'vax_rate': 0.58},
    'Brazil': {'pop': 214e6, 'vax_rate': 0.55},
    'UK':     {'pop': 67e6,  'vax_rate': 0.72},
    'Germany':{'pop': 83e6,  'vax_rate': 0.70},
    'France': {'pop': 67e6,  'vax_rate': 0.65},
    'Egypt':  {'pop': 104e6, 'vax_rate': 0.42},
    'Italy':  {'pop': 60e6,  'vax_rate': 0.69},
    'Russia': {'pop': 146e6, 'vax_rate': 0.50},
    'Japan':  {'pop': 126e6, 'vax_rate': 0.80},
}

dates = pd.date_range('2020-03-01', '2023-12-31', freq='W')
records = []

for country, info in countries.items():
    pop = info['pop']
    vax = info['vax_rate']
    cumulative_cases = 0
    cumulative_deaths = 0
    cumulative_vax = 0

    for i, date in enumerate(dates):
        # Simulate waves
        wave = (np.sin(i / 20) + 1.2) * (1 - cumulative_vax / pop * 0.8)
        new_cases  = max(0, int(np.random.poisson(pop * 0.0003 * wave)))
        new_deaths = max(0, int(new_cases * np.random.uniform(0.008, 0.025)))
        new_vax    = max(0, int(pop * vax / 100 * np.random.uniform(0.8, 1.2))) if date.year >= 2021 else 0

        cumulative_cases  += new_cases
        cumulative_deaths += new_deaths
        cumulative_vax    = min(cumulative_vax + new_vax, int(pop * vax))

        records.append({
            'Date':             date,
            'Country':          country,
            'Population':       int(pop),
            'New_Cases':        new_cases,
            'New_Deaths':       new_deaths,
            'Cumulative_Cases': cumulative_cases,
            'Cumulative_Deaths':cumulative_deaths,
            'Vaccinated':       cumulative_vax,
        })

df = pd.DataFrame(records)
df['Cases_per_Million']  = (df['Cumulative_Cases']  / df['Population'] * 1e6).round(1)
df['Deaths_per_Million'] = (df['Cumulative_Deaths'] / df['Population'] * 1e6).round(1)
df['Mortality_Rate']     = (df['Cumulative_Deaths'] / df['Cumulative_Cases'].replace(0, np.nan) * 100).round(3)
df['Vax_Rate_Pct']       = (df['Vaccinated'] / df['Population'] * 100).round(1)
df['Year']               = df['Date'].dt.year
df['Month']              = df['Date'].dt.to_period('M')

print(f'✅ Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Countries: {df["Country"].nunique()} | Date Range: {df["Date"].min().date()} → {df["Date"].max().date()}')
df.head()

## 3. 🔍 Data Cleaning & Overview

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())
print(f'\n=== Key Statistics ===')
latest = df.groupby('Country').last().reset_index()
print(latest[['Country','Cumulative_Cases','Cumulative_Deaths','Vaccinated','Vax_Rate_Pct']].to_string(index=False))

## 4. 📈 Visualizations

### 4.1 Cumulative Cases Over Time (Top Countries)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
colors = ['#E53935','#1E88E5','#43A047','#FB8C00','#8E24AA','#00ACC1','#F4511E','#6D4C41','#546E7A','#FFB300']

for (country, grp), color in zip(df.groupby('Country'), colors):
    ax.plot(grp['Date'], grp['Cumulative_Cases']/1e6, label=country, linewidth=2, color=color)

ax.set_title('Cumulative COVID-19 Cases Over Time', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Date'); ax.set_ylabel('Cumulative Cases (Millions)')
ax.legend(loc='upper left', fontsize=9, ncol=2)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.1f}M'))
plt.tight_layout()
plt.savefig('cumulative_cases.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: cumulative_cases.png')

### 4.2 Deaths per Million by Country

In [ ]:
latest = df.groupby('Country').last().reset_index().sort_values('Deaths_per_Million', ascending=True)

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(latest['Country'], latest['Deaths_per_Million'],
               color=plt.cm.Reds(np.linspace(0.3, 0.9, len(latest))))
ax.set_title('Total Deaths per Million Population', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Deaths per Million')
for bar, val in zip(bars, latest['Deaths_per_Million']):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('deaths_per_million.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: deaths_per_million.png')

### 4.3 Vaccination Progress Over Time

In [ ]:
vax_df = df[df['Date'].dt.year >= 2021]
fig, ax = plt.subplots(figsize=(14, 6))

for (country, grp), color in zip(vax_df.groupby('Country'), colors):
    ax.plot(grp['Date'], grp['Vax_Rate_Pct'], label=country, linewidth=2, color=color)

ax.set_title('Vaccination Rate Progress by Country (%)', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Date'); ax.set_ylabel('Vaccinated Population (%)')
ax.legend(loc='upper left', fontsize=9, ncol=2)
ax.axhline(60, color='gray', linestyle='--', linewidth=1, label='60% threshold')
plt.tight_layout()
plt.savefig('vaccination_progress.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: vaccination_progress.png')

### 4.4 Mortality Rate Heatmap (Country × Year)

In [ ]:
pivot = df.groupby(['Country','Year'])['Mortality_Rate'].mean().unstack()

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'Mortality Rate (%)'}, ax=ax)
ax.set_title('Average Mortality Rate (%) — Country × Year', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('mortality_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: mortality_heatmap.png')

### 4.5 New Cases Weekly Trend (2020-2023)

In [ ]:
global_weekly = df.groupby('Date')['New_Cases'].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(global_weekly['Date'], global_weekly['New_Cases']/1e6, alpha=0.3, color='#E53935')
ax.plot(global_weekly['Date'], global_weekly['New_Cases']/1e6, color='#E53935', linewidth=1.5)
ax.set_title('Global Weekly New Cases (All Countries Combined)', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Date'); ax.set_ylabel('New Cases (Millions)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.2f}M'))
plt.tight_layout()
plt.savefig('weekly_new_cases.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: weekly_new_cases.png')

### 4.6 Cases vs Vaccination Rate (Latest Data)

In [ ]:
latest = df.groupby('Country').last().reset_index()

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(latest['Vax_Rate_Pct'], latest['Cases_per_Million'],
                     s=latest['Population']/2e6, alpha=0.7,
                     c=latest['Deaths_per_Million'], cmap='YlOrRd')
plt.colorbar(scatter, ax=ax, label='Deaths per Million')

for _, row in latest.iterrows():
    ax.annotate(row['Country'], (row['Vax_Rate_Pct'], row['Cases_per_Million']),
                fontsize=9, ha='left', xytext=(4, 4), textcoords='offset points')

ax.set_title('Vaccination Rate vs Cases per Million', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Vaccination Rate (%)')
ax.set_ylabel('Cases per Million')
plt.tight_layout()
plt.savefig('vax_vs_cases.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: vax_vs_cases.png')

## 5. 📌 Key Insights Report

In [ ]:
latest = df.groupby('Country').last().reset_index()

print('=' * 50)
print('       🦠 COVID-19 GLOBAL INSIGHT REPORT')
print('=' * 50)
print(f'  📅 Analysis Period    : Mar 2020 – Dec 2023')
print(f'  🌍 Countries Covered  : {df["Country"].nunique()}')
print(f'  📊 Total Records      : {len(df):,}')
print()
print(f'  🔴 Highest Cases/M    : {latest.loc[latest["Cases_per_Million"].idxmax(),"Country"]}')
print(f'  💀 Highest Deaths/M   : {latest.loc[latest["Deaths_per_Million"].idxmax(),"Country"]}')
print(f'  💉 Highest Vax Rate   : {latest.loc[latest["Vax_Rate_Pct"].idxmax(),"Country"]} ({latest["Vax_Rate_Pct"].max():.1f}%)')
print(f'  💉 Lowest Vax Rate    : {latest.loc[latest["Vax_Rate_Pct"].idxmin(),"Country"]} ({latest["Vax_Rate_Pct"].min():.1f}%)')
print(f'  📉 Lowest Mortality   : {latest.loc[latest["Mortality_Rate"].idxmin(),"Country"]}')
print('=' * 50)

## 6. 💾 Export Data

In [ ]:
df.to_csv('covid19_data.csv', index=False)
print(f'✅ Exported → covid19_data.csv ({len(df):,} rows)')

## 7. 📝 Conclusion

### Key Findings:
- Countries with **higher vaccination rates** show significantly lower mortality in 2022–2023
- **Mortality rates** declined across all countries as vaccines rolled out in 2021
- Clear **wave patterns** visible in weekly new cases, peaking in Winter months
- **Japan** achieved the highest vaccination rate with the lowest mortality rate
- **Egypt** had the lowest vaccination rate among analyzed countries

### Recommendations:
1. Accelerate vaccination programs in lower-coverage countries
2. Strengthen healthcare capacity before Winter wave peaks
3. Use mortality trend data to allocate medical resources proactively
